In [1]:
import sys
import os
import torch

In [2]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment

2025-02-22 17:06:06,998 - INFO - NumExpr defaulting to 8 threads.


In [3]:
from torch_geometric.data import Data
def print_subgraph_stats(data: Data, name: str = "Subgraph"):
    """
    Prints basic statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object
        name: Name/identifier for the subgraph (default="Subgraph")
    """
    print(f"{name} stats:")
    print(f"Number of nodes: {data.num_nodes}")
    print(f"Number of edges: {data.edge_index.size(1)}")
    print(f"Number of features: {data.x.size(1)}")
    print(f"Number of classes: {len(torch.unique(data.y))}")
    print(f"Number of training nodes: {data.train_mask.sum().item()}")
    print(f"Number of validation nodes: {data.val_mask.sum().item()}")
    print(f"Number of test nodes: {data.test_mask.sum().item()}")
    print(f"Zero feature vectors: {(data.x.sum(dim=1) == 0).sum().item()}")
    print("-------------------")
def print_node_indices(subgraph):
    """
    Print the indices of training, validation, and test nodes in the given subgraph.

    Args:
        subgraph: A graph or subgraph object that contains train_mask, val_mask, and test_mask attributes.
    """
    print(f"Training nodes in the subgraph: {subgraph.train_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of training nodes: {subgraph.train_mask.sum().item()}")
    print(f"Validation nodes in the subgraph: {subgraph.val_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of validation nodes: {subgraph.val_mask.sum().item()}")
    print(f"Test nodes in the subgraph: {subgraph.test_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of test nodes: {subgraph.test_mask.sum().item()}")
# Example usage:
# print_subgraph_stats(subgraph, "Original subgraph")
# print_subgraph_stats(expanded_subgraph, "Expanded subgraph")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
from run import load_and_split_with_khop
data, dataset, clients_data, test_data,  = load_and_split_with_khop("Cora", device, num_clients=10, beta=0.5, hop=1)
# review data for client_0
sg = clients_data[0]
print_subgraph_stats(sg, "Client 0")
print_node_indices(sg)


Client 0 stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 241
-------------------
Training nodes in the subgraph: [2, 3, 5, 9, 13, 14, 15, 18, 20, 21, 22]
Number of training nodes: 11
Validation nodes in the subgraph: [23, 28, 32, 33, 34, 38, 42, 43, 54, 58, 61, 65, 71, 75, 82, 86]
Number of validation nodes: 16
Test nodes in the subgraph: [224, 226, 232, 233, 235, 244, 252, 256, 260, 262, 263, 265, 268, 269, 271, 273, 283, 285, 295, 298, 299, 303, 306, 310, 313, 316, 319, 321, 325, 333, 337, 339, 341, 343]
Number of test nodes: 34


In [5]:
# load with fp and do the same thing
from run import load_and_split_with_feature_prop
data, dataset, clients_data, test_data,  = load_and_split_with_feature_prop("Cora", device, num_clients=10, beta=0.5, hop=1)
# review data for client_0
sg = clients_data[0]
print_subgraph_stats(sg, "Client 0")
print_node_indices(sg)


Client 0 stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 0
-------------------
Training nodes in the subgraph: [2, 3, 5, 9, 13, 14, 15, 18, 20, 21, 22]
Number of training nodes: 11
Validation nodes in the subgraph: [23, 28, 32, 33, 34, 38, 42, 43, 54, 58, 61, 65, 71, 75, 82, 86]
Number of validation nodes: 16
Test nodes in the subgraph: [224, 226, 232, 233, 235, 244, 252, 256, 260, 262, 263, 265, 268, 269, 271, 273, 283, 285, 295, 298, 299, 303, 306, 310, 313, 316, 319, 321, 325, 333, 337, 339, 341, 343]
Number of test nodes: 34


In [6]:
# examien a singraph from the kop thing
sg = clients_data[0]
# check number of nodes in the subgraph
print(f"Number of nodes in the subgraph: {sg.num_nodes}")
# check number of edges in the subgraph
print(f"Number of edges in the subgraph: {sg.edge_index.shape[1]}")
# check features of the subgraph
print(f"Features of the subgraph: {sg.x.shape}")
# check number of training nodes
print(f"Number of training nodes: {sg.train_mask.sum()}")
# check number of validation nodes
print(f"Number of validation nodes: {sg.val_mask.sum()}")
# check number of test nodes
print(f"Number of test nodes: {sg.test_mask.sum()}")
# number of nodes with zero feature vectors sum for each node then fidn the ones whose sum is zero
zero_features = (sg.x.sum(dim=1) == 0).sum()
print(f"Number of nodes with zero feature vectors: {zero_features}")













Number of nodes in the subgraph: 344
Number of edges in the subgraph: 1056
Features of the subgraph: torch.Size([344, 1433])
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Number of nodes with zero feature vectors: 0


In [7]:
import torch

print(f"Number of GPUs available: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU available")

Number of GPUs available: 0
No GPU available


In [8]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment
import os
import ray
ray.shutdown()

if __name__ == "__main__":
    # Define all options in a list
    # data_loading_options = ["split_dataset", "split_dataset_with_khop", "split_dataset_with_feature_prop"]
    data_loading_options = ["split_dataset_with_khop", "split_dataset_with_feature_prop", "split_dataset"]
    model_types = ["GCN"]
        
    # Load configuration once since it's common for all runs
    clients_num, beta, cfg = load_configuration()

    clients_num = 10

    # Main results directory
    main_dir = 'results'
    sub_dir = 'More_epochs_results_monte_carlo'
    # main_dir = 'New_results'
    
    dataset_name = "Cora"  # You can change this if you have different datasets

    # Loop over all combinations of data_loading_option and model_type
    for data_loading_option in data_loading_options:
        for model_type in model_types:
            # Create a structured directory for each experiment
            result_name = f"{dataset_name}_{data_loading_option}_{model_type}"
            results_dir = os.path.join(main_dir, sub_dir, result_name)

            # Create the directory if it doesn't exist
            os.makedirs(results_dir, exist_ok=True)

            print(f"Running experiment with data_loading_option: {data_loading_option}, model_type: {model_type}")
            result = main_experiment(clients_num, beta, data_loading_option, model_type, cfg, dataset_name=dataset_name, hop=2)
            
            # Replace literal '\n' with actual newline characters if necessary
            result_with_newlines = result.replace('\\n', '\n')
            
            # Create a unique filename for each combination
            filename = f'results_{dataset_name}_{data_loading_option}_{model_type}.txt'
            filepath = os.path.join(results_dir, filename)
            
            # Write the result to the text file
            with open(filepath, 'w') as f:
                f.write(result_with_newlines)
            
            print(f"Results saved to {filepath}\n")

Running experiment with data_loading_option: split_dataset_with_khop, model_type: GCN
DEVICE: cpu


2025-02-22 17:06:37,686	INFO worker.py:1770 -- Started a local Ray instance.


data_loading_option: split_dataset_with_khop
Data loaded
Device is cpu
Cora()
10


(FLClient pid=26696) 2025-02-22 17:06:48,467 - INFO - Epoch   0| Train Loss: 1.953| Train Accuracy: 0.000
(FLClient pid=26696) 2025-02-22 17:06:48,475 - INFO - Epoch   1| Train Loss: 1.880| Train Accuracy: 0.312
(FLClient pid=26696) 2025-02-22 17:06:48,484 - INFO - Epoch   2| Train Loss: 1.823| Train Accuracy: 0.312
(FLClient pid=26696) 2025-02-22 17:06:48,515 - INFO - Epoch   4| Validation Loss: 2.016, Validation Accuracy: 0.105


Training done
Round 1
Train Loss: 2.138, Train Accuracy: 0.357
Train Loss: 2.016, Train Accuracy: 0.375
Train Loss: 2.072, Train Accuracy: 0.667
Train Loss: 1.789, Train Accuracy: 0.300
Train Loss: 1.940, Train Accuracy: 0.357
Train Loss: 1.760, Train Accuracy: 0.267
Train Loss: 1.957, Train Accuracy: 0.467
Train Loss: 2.037, Train Accuracy: 0.214
Train Loss: 1.951, Train Accuracy: 0.643
Train Loss: 1.897, Train Accuracy: 0.538
Round 2
Train Loss: 2.114, Train Accuracy: 0.357
Train Loss: 2.008, Train Accuracy: 0.375
Train Loss: 2.055, Train Accuracy: 0.733
Train Loss: 1.747, Train Accuracy: 0.400
Train Loss: 1.931, Train Accuracy: 0.357
Train Loss: 1.713, Train Accuracy: 0.267
Train Loss: 1.946, Train Accuracy: 0.667
Train Loss: 2.025, Train Accuracy: 0.500
Train Loss: 1.937, Train Accuracy: 0.786
Train Loss: 1.894, Train Accuracy: 0.769
Round 3
Train Loss: 2.098, Train Accuracy: 0.357
Train Loss: 1.992, Train Accuracy: 0.438
Train Loss: 2.035, Train Accuracy: 0.800
Train Loss: 1.729, 

...

data_loading_option: split_dataset
Data loaded
Device is cpu
Cora()
10


(FLClient pid=27324) 2025-02-22 17:14:38,399 - INFO - Epoch   0| Train Loss: 1.895| Train Accuracy: 0.214
(FLClient pid=27324) 2025-02-22 17:14:38,423 - INFO - Epoch   4| Validation Loss: 2.060, Validation Accuracy: 0.082


Training done
Round 1
Train Loss: 2.060, Train Accuracy: 0.786
Train Loss: 2.071, Train Accuracy: 0.875
Train Loss: 2.167, Train Accuracy: 0.933
Train Loss: 1.819, Train Accuracy: 1.000
Train Loss: 1.857, Train Accuracy: 1.000
Train Loss: 1.761, Train Accuracy: 0.933
Train Loss: 2.008, Train Accuracy: 1.000
Train Loss: 2.012, Train Accuracy: 1.000
Train Loss: 1.903, Train Accuracy: 1.000
Train Loss: 1.923, Train Accuracy: 0.692
Round 2
Train Loss: 1.938, Train Accuracy: 0.929
Train Loss: 2.016, Train Accuracy: 0.938
Train Loss: 2.149, Train Accuracy: 0.933
Train Loss: 1.694, Train Accuracy: 1.000
Train Loss: 1.743, Train Accuracy: 1.000
Train Loss: 1.647, Train Accuracy: 1.000
Train Loss: 1.934, Train Accuracy: 1.000
Train Loss: 1.963, Train Accuracy: 1.000
Train Loss: 1.829, Train Accuracy: 1.000
Train Loss: 1.845, Train Accuracy: 0.923
Round 3
Train Loss: 1.813, Train Accuracy: 1.000
Train Loss: 1.935, Train Accuracy: 1.000
Train Loss: 2.068, Train Accuracy: 1.000
Train Loss: 1.611, 

(FLClient pid=27333) 2025-02-22 17:14:42,482 - INFO - Epoch   4| Train Loss: 0.023| Train Accuracy: 1.000 [repeated 1499x across cluster]
(FLClient pid=27333) 2025-02-22 17:14:42,484 - INFO - Epoch   4| Validation Loss: 1.375, Validation Accuracy: 0.556 [repeated 299x across cluster]


Round 5 is complete
The global test results: [0.783, 0.782, 0.778, 0.779, 0.786]
The client test results: [0.631180490668553, 0.6316521859152602, 0.6280021904009458, 0.6375691832532074, 0.6276436791828252]
The average global test results: 0.7816
The average client test results: 0.6312095458841582
The standad deviation global is: 0.002870540018881467
The standad deviation client is: 0.0035677526931070635
Results saved to results/More_epochs_results_monte_carlo/Cora_split_dataset_GCN/results_Cora_split_dataset_GCN.txt



### Tests